In [1]:
import csv
import os
import pickle
import numpy as np
import pandas as pd
from pubchempy import *
from tqdm.notebook import tqdm
from pipeline.features import load_drug_smile, save_cell_meth_matrix, save_cell_mut_matrix, save_cell_oge_matrix
from utils import *

In [3]:
def read_pkl(path):
  with open(path, "rb") as f:
    return pickle.load(f)
def save_pkl(path, obj):
  with open(path, "wb") as f:
    return pickle.dump(obj)

In [4]:
# imports consolidated above

In [5]:
# imports consolidated above

In [6]:
cell_dict_mut, cell_feature_mut = save_cell_mut_matrix()
cell_dict_meth = save_cell_meth_matrix()
drug_dict, drug_smile, smile_graph = load_drug_smile()

In [7]:
drug_pt_df = pd.read_csv("data/drug_physic_toxic_df.csv")
drug_pt_df = drug_pt_df.set_index("Drug")

In [8]:
drug_pt_df.loc["5-FU"]

0       0.000000
1       0.000000
2       0.000000
3       0.000000
4       0.000000
          ...   
3073    0.000000
3074    0.000000
3075    0.007499
3076    0.086820
3077    0.000000
Name: 5-FU, Length: 3078, dtype: float64

In [9]:
# imports consolidated above

In [10]:
os.listdir("data/split_data/")

['all_test', 'mix_data', 'blind_cell', 'blind_drug', '.DS_Store']

In [11]:
data_names = ALL_DATA_SPLIT_FILES

In [12]:
def subype_to_array(subtype):
    subtypes = ['aero_digestive_tract', 'blood', 'bone', 'breast',
       'digestive_system', 'kidney', 'lung', 'nervous_system', 'pancreas',
       'skin', 'soft_tissue', 'thyroid', 'urogenital_system']
    temp = np.zeros(len(subtypes))
    temp[subtypes.index(subtype)] = 1
    return temp

In [13]:
def create_data(data_name, root):
    global_path = data_split_path
    temp_dir = os.path.join(data_split_path, "temp")
    os.makedirs(temp_dir, exist_ok=True)
    with open(global_path + data_name+".csv", newline="") as f:
        reader = csv.reader(f)
        next(reader)
        temp_data = []
        for item in reader:
            drug_1 = str(item[0])
            drug_2 = str(item[1])
            cell = int(item[3])
            loewe = float(item[4])
            temp_data.append((drug_1, drug_2, cell, loewe))
    xd_1 = []
    xd_pt_1 = []
    xd_2 = []
    xd_pt_2 = []
    xc_mut = []
    xc_meth = []
    xc_ge = []    
    y = []
    lst_drug = []
    lst_cell = []
#     random.shuffle(temp_data)
    for data in temp_data:
        drug_1, drug_2, cell, loewe = data
        if (
            drug_1 in drug_dict and drug_2 in drug_dict
            and drug_1 in drug_pt_df.index and drug_2 in drug_pt_df.index
            and cell in cell_dict_ge and cell in cell_dict_mut and cell in cell_dict_meth
        ):
            xc_mut.append(cell_feature_mut[cell_dict_mut[cell]])
            xc_meth.append(cell_dict_meth[cell])
            xc_ge.append(cell_dict_ge[cell])
            xd_1.append(drug_smile[drug_dict[drug_1]])
            xd_pt_1.append(drug_pt_df.loc[drug_1])
            xd_2.append(drug_smile[drug_dict[drug_2]])
            xd_pt_2.append(drug_pt_df.loc[drug_2])
            y.append(loewe)
            lst_drug.append((drug_1, drug_2))
            lst_cell.append(cell)
    with open(os.path.join(temp_dir, 'drug_dict_' + data_name), 'wb') as fp:
        pickle.dump(drug_dict, fp)
        
    print(len(lst_drug))
    print(len(lst_cell))

    if len(y) == 0:
        print(f"skip {data_name}: no valid samples")
        return None

    xd_1, xd_pt_1, xd_2, xd_pt_2, xc_mut, xc_meth, xc_ge, y = np.asarray(xd_1), np.asarray(xd_pt_1), np.asarray(xd_2), np.asarray(xd_pt_2), np.asarray(xc_mut), np.asarray(xc_meth), np.asarray(xc_ge), np.asarray(y)

    with open(os.path.join(temp_dir, 'list_drug_' + data_name), 'wb') as fp:
        pickle.dump(lst_drug, fp)
        
    with open(os.path.join(temp_dir, 'list_cell_' + data_name), 'wb') as fp:
        pickle.dump(lst_cell, fp)

    dataset = 'GDSC'
    print('preparing ', dataset + '_train.pkl in pytorch format!')
    data = TestbedDataset(root=root, dataset=dataset+"_"+data_name, xd_1=xd_1, xd_pt_1=xd_pt_1, xd_2=xd_2, xd_pt_2=xd_pt_2, xt_mut=xc_mut, xt_meth=xc_meth, xt_ge=xc_ge, y=y, smile_graph=smile_graph)

count = 1


In [14]:
data_split_path_temp = "data/split_data/"
root_temp = "data/split_data/"
data_sets = ["all_test/"]
for data_set in data_sets:
    print(f"_________________________{data_set}_____________________________")
    count = 1
    data_split_path = data_split_path_temp+data_set
    root = root_temp + data_set
    ge_path = data_split_path + "ge_process.pkl"
    ge = pd.read_pickle(ge_path) if os.path.exists(ge_path) else pd.read_csv(data_split_path+"ge_process.csv")
    ge = ge.rename(columns={"Unnamed: 0":"Cosmic sample Id"})
    cell_dict_ge = save_cell_oge_matrix()
    for data_name in tqdm(data_names):
      csv_path = os.path.join(data_split_path, data_name + ".csv")
      if not os.path.exists(csv_path):
        continue
      print(count, ":", data_name)
      create_data(data_name, root)
      count += 1

_________________________all_test/_____________________________


  0%|          | 0/1018 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

1 : mix_test
1440
1440
preparing  GDSC_train.pkl in pytorch format!
Pre-processed data found: data/split_data/all_test/processed/GDSC_mix_test.pkl, loading ...
2 : train_dc
11512
11512
preparing  GDSC_train.pkl in pytorch format!
Pre-processed data found: data/split_data/all_test/processed/GDSC_train_dc.pkl, loading ...
3 : test_dc
8968
8968
preparing  GDSC_train.pkl in pytorch format!
Pre-processed data data/split_data/all_test/processed/GDSC_test_dc.pkl not found, doing pre-processing...


/home/f/GraOmicSynergy/utils.py:104: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  GCNData.target_mut = torch.FloatTensor([target_mut])


Graph construction done. Saving to file.
4 : val_dc
8964
8964
preparing  GDSC_train.pkl in pytorch format!
Pre-processed data data/split_data/all_test/processed/GDSC_val_dc.pkl not found, doing pre-processing...
Graph construction done. Saving to file.
